# FWER calibration for skchange detectors

A detector fires when a penalised score exceeds zero. Default (BIC-style) penalties
are calibrated to the *number of observations and features* — not to a specific
false-alarm rate. On short series they can produce far more alarms than intended.

**Goal.** Tune a detector's `penalty_scale` so the **family-wise error rate** (FWER)
— the probability of flagging *at least one* change on change-free data — matches a
target `level`.

**Mechanism.** For each null (change-free) sample, find the smallest `penalty_scale`
that suppresses every detection; the calibrated scale is the `(1 − level)` quantile
of those critical scales.

**Calibration strategies (set automatically per detector):**

- `"max_score"` (SBS, MovingWindow, CircularBinSeg, CAPA): closed-form
  `c_b = max(S) / base`. Exact — one fit per null sample.

- `"detection_count"` (PELT and the default): bisect `penalty_scale` until detections
  hit zero. PELT uses this because it optimises jointly over all changepoint sets —
  the single-split score underestimates the true critical penalty. ~15-25 fits per
  null sample.

## 0. Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from skchange.new_api.detectors import (
    PELT,
    SeededBinarySegmentation,
)
from skchange.new_api.tuning import (
    CalibratedDetector,
    calibrate_penalty_scale,
)

# ---- settings (notebook runs in ~1-2 minutes) ----
N, P = 80, 2  # series length
LEVEL = 0.1  # target FWER
N_SIMS = 1000  # Monte Carlo draws for calibration
N_EVAL = 1000  # independent null series for empirical-FWER check
RNG = np.random.default_rng(0)


def null_X(n=N, p=P, seed=0):
    return np.random.default_rng(seed).normal(size=(n, p))


def empirical_fwer(detector, n_eval=N_EVAL, n=N, p=P, seed=99):
    """Fraction of change-free series on which detector flags >= 1 change."""
    rng = np.random.default_rng(seed)
    fa = sum(
        1
        for _ in range(n_eval)
        if len(detector.fit(X := rng.normal(size=(n, p))).predict_changepoints(X)) > 0
    )
    return fa / n_eval

## 1. The problem: default penalties are miscalibrated

SBS fires on ~81% of null series of length 80 — well above the 5% target.
PELT happens to be near 5% for this length, but neither is targeting a stated level.

In [2]:
for name, det in [
    ("SBS (default)", SeededBinarySegmentation()),
    ("PELT (default)", PELT()),
]:
    fwer = empirical_fwer(det)
    print(f"{name:22s}  FWER = {fwer:.1%}  (target {LEVEL:.0%})")

SBS (default)           FWER = 83.4%  (target 10%)
PELT (default)          FWER = 3.0%  (target 10%)


## 2. Calibration in one call

We use the Gaussian sampler (`sampler="gaussian"`) which draws i.i.d. N(0,1)
null samples of the same shape as `X`. It gives tighter estimates than the
permutation sampler for short series when the null distribution is Gaussian.

In [3]:
X_ref = null_X()

scale_sbs = calibrate_penalty_scale(
    SeededBinarySegmentation(),
    X_ref,
    sampler="gaussian",
    level=LEVEL,
    n_simulations=N_SIMS,
    random_state=42,
)
scale_pelt = calibrate_penalty_scale(
    PELT(),
    X_ref,
    sampler="gaussian",
    level=LEVEL,
    n_simulations=N_SIMS,
    random_state=42,
)

print(f"Calibrated penalty_scale — SBS:  {scale_sbs:.3f}")
print(f"Calibrated penalty_scale — PELT: {scale_pelt:.3f}")

fwer_sbs = empirical_fwer(
    SeededBinarySegmentation().set_params(penalty_scale=scale_sbs)
)
fwer_pelt = empirical_fwer(PELT().set_params(penalty_scale=scale_pelt))
print(f"\nEmpirical FWER — SBS:  {fwer_sbs:.1%}  (target {LEVEL:.0%})")
print(f"Empirical FWER — PELT: {fwer_pelt:.1%}  (target {LEVEL:.0%})")

Calibrated penalty_scale — SBS:  1.403
Calibrated penalty_scale — PELT: 0.848

Empirical FWER — SBS:  8.9%  (target 10%)
Empirical FWER — PELT: 8.8%  (target 10%)


## 3. `CalibratedDetector` — the meta-estimator

`CalibratedDetector` wraps any supported detector, runs calibration inside `fit`,
and stores the calibrated detector as `detector_`. It is a standard sklearn estimator.

In [4]:
cal = CalibratedDetector(
    PELT(), sampler="gaussian", level=LEVEL, n_simulations=N_SIMS, random_state=42
).fit(X_ref)

print(f"penalty_scale_ = {cal.penalty_scale_:.3f}")

# Detect on data with a real change at t=40
X_change = np.vstack([RNG.normal(0, 1, (40, P)), RNG.normal(3, 1, (40, P))])
cps = cal.predict_changepoints(X_change)
print(f"Changepoints detected near t=40: {cps}")

penalty_scale_ = 0.848
Changepoints detected near t=40: [40]


## 4. Money plot: empirical FWER vs nominal level

If calibration is correct the curve lies on the diagonal. Both detectors should
track it well.

> **Runtime note.** SBS uses `"max_score"` (one fit per null sample, very fast).
> PELT uses `"detection_count"` (bisection, ~15-25 fits per null sample). This cell
> takes about 1 minute.

In [ ]:
sweep_levels = [0.01, 0.05, 0.10, 0.20, 0.30]
sweep_detectors = {"SBS": SeededBinarySegmentation, "PELT": PELT}

results = {name: [] for name in sweep_detectors}

for name, cls in sweep_detectors.items():
    for lv in sweep_levels:
        scale = calibrate_penalty_scale(
            cls(),
            X_ref,
            sampler="gaussian",
            level=lv,
            n_simulations=N_SIMS,
            random_state=42,
        )
        fwer = empirical_fwer(cls().set_params(penalty_scale=scale))
        results[name].append(fwer)
    print(f"{name}: empirical FWER = {[f'{f:.0%}' for f in results[name]]}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 0.35], [0, 0.35], "k--", lw=1, label="ideal (y = x)")
for (name, fwers), mk in zip(results.items(), ["o", "s"]):
    ax.plot(sweep_levels, fwers, mk + "-", label=name)
ax.set_xlabel("nominal level")
ax.set_ylabel("empirical FWER")
ax.set_title("Calibration curve")
ax.legend()
plt.tight_layout()
plt.show()

SBS: empirical FWER = ['1%', '5%', '9%', '21%', '30%']


## 5. Clean vs contaminated calibration data

If the series passed to `calibrate_penalty_scale` **contains a real change**, the
permuted null samples inherit inflated scores → the penalty is pushed too high →
over-conservative detection. Pass a separate clean series via `X_calib` to fix this.

Note: this effect is specific to the `"permutation"` sampler (which resamples from
`X`). The Gaussian sampler always draws fresh N(0,1) data and is unaffected by
contamination in `X`.

In [ ]:
rng2 = np.random.default_rng(7)
X_dirty = np.vstack(
    [rng2.normal(0, 1, (40, P)), rng2.normal(4, 1, (40, P))]
)  # has a change!
X_clean = rng2.normal(0, 1, (200, P))  # genuinely change-free

scale_dirty = calibrate_penalty_scale(
    SeededBinarySegmentation(),
    X_dirty,
    sampler="permutation",
    n_simulations=N_SIMS,
    random_state=0,
)
scale_clean = calibrate_penalty_scale(
    SeededBinarySegmentation(),
    X_dirty,
    X_calib=X_clean,
    sampler="permutation",
    n_simulations=N_SIMS,
    random_state=0,
)

print(f"scale from contaminated X:  {scale_dirty:.3f}  (inflated -> over-conservative)")
print(f"scale with clean X_calib:   {scale_clean:.3f}  (correct)")

## 6. Scratch

Experiment here — try other detectors (`MovingWindow`, `CAPA`), different `level`
values, series lengths, or the permutation sampler.

In [ ]:
# your experiments here